In [ ]:
# drive erisimi icin
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Detectron2 kurulumu
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
# Dataset'i Drive'dan colab'in icerisine cikarmak icin
!unzip "/content/drive/MyDrive/Mask_RCNN_Project/dataset_clean.zip" -d "/content/dataset"

In [ ]:
# ============================================================
# 1) KÜTÜPHANELER
# ============================================================
import os
import json # json formatindaki verileri okumak,yazmak icin
import random # random secimler yapmak icin
import cv2
import matplotlib.pyplot as plt # grafik ve gorsellestirme icin

from detectron2.engine import DefaultTrainer, DefaultPredictor # modelin egitilmesi ve egitim bittikten sonra sonuc almak icin
from detectron2.config import get_cfg
from detectron2 import model_zoo # bazı hazır model/configuration dosyalarina erisim icin
from detectron2.data import MetadataCatalog, DatasetCatalog, build_detection_test_loader
# metadata:sinif isimleri gibi bilgiler sinif id vs, DatasetCatalog: dataseti modele tanitmak icin bilgiler burada, build_detection_test_loader: test datasetinden goruntuleri modele verebilmek icin
from detectron2.data.datasets import register_coco_instances # .cocoijson formatindaki dataseti Detectron2 ye kaydetmek icin
from detectron2.evaluation import COCOEvaluator, inference_on_dataset # model sonuclarini metriklerle degerlendirmek icin
from detectron2.utils.visualizer import Visualizer, ColorMode # bbox, segmentation mask gibi sonuclari goruntuye cizmek icin
from google.colab.patches import cv2_imshow # colab ortaminda goruntu gostermek icin

In [ ]:
# ============================================================
# 2) VERİ SETLERINI KAYDET
# ============================================================
# Eski dataset kayitlarini temizle
DatasetCatalog.clear()
MetadataCatalog.clear()
# COCO formatindaki dataseti Detectron2'ye kaydet. datasete verilen isim,ekstra metadata yok,bbox segmantation verilerinin oldugu dosya yolu,goruntulerin bulundugu klasorun yolu
register_coco_instances("my_dataset_train", {}, "/content/dataset/dataset_clean/train/train_temiz_annotations.coco.json", "/content/dataset/dataset_clean/train")
register_coco_instances("my_dataset_valid", {}, "/content/dataset/dataset_clean/valid/valid_temiz_annotations.coco.json", "/content/dataset/dataset_clean/valid")
register_coco_instances("my_dataset_test",  {}, "/content/dataset/dataset_clean/test/test_temiz_annotations.coco.json",  "/content/dataset/dataset_clean/test")

# dataseti katalogdan getir.
DatasetCatalog.get("my_dataset_train")
DatasetCatalog.get("my_dataset_valid")
DatasetCatalog.get("my_dataset_test")

kategori_sayisi = len(MetadataCatalog.get("my_dataset_train").thing_classes) # datasetteki sinif sayisi
print("Sınıflar:", MetadataCatalog.get("my_dataset_train").thing_classes) # dataset sinif isimleri yazdirilir. thing_classes ile siniflara erisilir.
print("Sınıf sayısı:", kategori_sayisi)

In [ ]:
# ============================================================
# 3) CONFIG (EGITIM AYARLARI)
# ============================================================
cfg = get_cfg() # Detectron2'nin varsayilan nesnesi oluturulur ve cfg degiskenine atanir.

# modelin hangi mimariyi ve varsayilan ayarlari kullanacagini belirle.(model_zoo dan belirtilen modelin configuration dosyasi bulunur.)
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
# bulunan config. dosyasindaki ayarlar cfg icerisine atilir.
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")

cfg.DATASETS.TRAIN = ("my_dataset_train",)
cfg.DATASETS.TEST = ("my_dataset_valid",)

cfg.DATALOADER.NUM_WORKERS = 2 # verinin hazirlanip modele aktarilmasi icin worker sayisi.

cfg.SOLVER.IMS_PER_BATCH = 2                   # her iterasyonda modele 2 adet goruntu verilecek.
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 1000
cfg.SOLVER.STEPS = []                          # learning rate'in belirli iterasyonlarda dusurulmesini saglayan liste. bos birakildi.
cfg.SOLVER.CHECKPOINT_PERIOD = 200             # her 200 iterasyonda model kaydet

cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128 # RPN'nin cizdigi taslak aday kutularindan kac tanesinin kullanilacagini belirler.
cfg.MODEL.ROI_HEADS.NUM_CLASSES = kategori_sayisi # sinif sayisini modele verdik.

cfg.TEST.EVAL_PERIOD = 100                     # her 100 iterasyonda bir degerlendirme ygap.

cfg.OUTPUT_DIR = "/content/drive/MyDrive/Mask_RCNN_Project/output_maskrcnn"    # outputlar icin kulanilacak klasoru belirliyoruz.
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print("Config hazır. Sınıf sayısı:", cfg.MODEL.ROI_HEADS.NUM_CLASSES)

In [ ]:
# ============================================================
# 4) PERİYODİK DEĞERLENDİRME YAPAN TRAINER
# normalde training sirasinda valid seti uzerinde otomatik degerendirme yapamayan DefaultTrainer'a nasil degerlendirme yapacagini ogretiyoruz.
# {"iteration": 100, "segm/AP": 4.2, "segm/AP50": 12.1, "bbox/AP": 5.8, ...} seklinde tutulur.
# default olarakda 20 iterasyonda  bir loss loglar.
# ============================================================
class CocoTrainer(DefaultTrainer): #DefaultTrainer Detectron2'nin hazir trainer sinifinda kalitim aliniyor ancak bazi davranislari degisecek.
# DefaultTrainer yaptigi seyler : veri yukleme,optimizer,loglama,checkpoint kaydetme vs
    @classmethod # override etme
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "coco_eval") # ciktinin kaydedilecegi yer.
            os.makedirs(output_folder, exist_ok=True)                 # dosya yoksa olustur
        return COCOEvaluator(dataset_name, output_dir=output_folder)  # COCOEvalutor kullan.(Metrikleri hesaplar.)

In [ ]:
# ============================================================
# 5) EĞİTİMİ BAŞLAT
# ============================================================
trainer = CocoTrainer(cfg)
trainer.resume_or_load(resume=False)

print("Eğitim başlıyor...")
trainer.train()
print("Eğitim tamamlandı! Model:", os.path.join(cfg.OUTPUT_DIR, "model_final.pth"))

In [ ]:
# ============================================================
# 6) LOSS VE AP GRAFİKLERİ
# ============================================================
metrics_path = os.path.join(cfg.OUTPUT_DIR, "metrics.json")
with open(metrics_path, "r") as f:
    satirlar = [json.loads(l) for l in f]

def metrik_cek(anahtar):
    xs, ys = [], []
    for s in satirlar:
        if anahtar in s:
            xs.append(s["iteration"])
            ys.append(s[anahtar])
    return xs, ys

fig, axs = plt.subplots(2, 2, figsize=(14, 10))

it, val = metrik_cek("total_loss")
axs[0, 0].plot(it, val); axs[0, 0].set_title("Total Loss"); axs[0, 0].set_xlabel("iterasyon")

it, cls_loss = metrik_cek("loss_cls")
_, mask_loss = metrik_cek("loss_mask")
axs[0, 1].plot(it, cls_loss, label="loss_cls")
axs[0, 1].plot(it, mask_loss, label="loss_mask")
axs[0, 1].set_title("Classification / Mask Loss"); axs[0, 1].legend()

it, box_loss = metrik_cek("loss_box_reg")
axs[1, 0].plot(it, box_loss, color="orange"); axs[1, 0].set_title("Box Regression Loss")

it_ap, ap = metrik_cek("segm/AP")
if ap:
    axs[1, 1].plot(it_ap, ap, marker="o", color="green"); axs[1, 1].set_title("Validation Segmentation AP")
else:
    axs[1, 1].text(0.5, 0.5, "AP verisi henüz yok", ha="center")

plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, "egitim_grafikleri.png"))
plt.show()

In [ ]:
# ============================================================
# 7) FİNAL DEĞERLENDİRME (VALID + TEST) — sayısal AP tablosu
# ============================================================
# Egitilen son modelin agirliklarini kullan.
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5 # confidence(guven esigi) burada %50

predictor = DefaultPredictor(cfg)

for split_adi in ["my_dataset_valid", "my_dataset_test"]:
    print(f"\n=== {split_adi} sonuçları ===")
    evaluator = COCOEvaluator(split_adi, output_dir=os.path.join(cfg.OUTPUT_DIR, f"eval_{split_adi}")) #tahminleri gercek etiketlerle karsilastirip AP hesasaplama gorevini ustlenir
    val_loader = build_detection_test_loader(cfg, split_adi) # fotograflari uygun boyutlara getirir ve modele batchler halinde vermek icin veri hatti olusturur.
    print(inference_on_dataset(predictor.model, val_loader, evaluator)) # modeli o split'teki her resimde çalıştırıyor, her tahmini gerçek annotation'la kıyaslıyor, sonunda AP/AP50/AP75 tablosunu hesaplayıp döndürüyor.

In [ ]:
veri_olan = [(i, isim) for i, isim in enumerate(sinif_isimleri)
             if test_class_gorselleri[i] or valid_class_gorselleri[i] or train_class_gorselleri[i]]

# Rastgele bir class sec
i, sinif_adi = random.choice(veri_olan)

if test_class_gorselleri[i]:
    adaylar, kaynak_split = test_class_gorselleri[i], "test"
elif valid_class_gorselleri[i]:
    adaylar, kaynak_split = valid_class_gorselleri[i], "valid"
else:
    adaylar, kaynak_split = train_class_gorselleri[i], "train"

d   = random.choice(adaylar)
img = cv2.imread(d["file_name"])

fig, axs = plt.subplots(1, 2, figsize=(14, 6))

# SOL — Gerçek etiket
v_gt   = Visualizer(img[:, :, ::-1], metadata=test_metadata, scale=0.7)
out_gt = v_gt.draw_dataset_dict(d)
axs[0].imshow(out_gt.get_image())
axs[0].set_title(f"{sinif_adi} — Gerçek Etiket ({kaynak_split})")
axs[0].axis("off")

# SAĞ — Model tahmini
outputs  = predictor(img)
v_pred   = Visualizer(img[:, :, ::-1], metadata=test_metadata, scale=0.7)
out_pred = v_pred.draw_instance_predictions(outputs["instances"].to("cpu"))
axs[1].imshow(out_pred.get_image())
axs[1].set_title(f"{sinif_adi} — Model Tahmini")
axs[1].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, "temsili_karsilastirma.png"))
plt.show()